# 05 Dashboard data preparation

This notebook prepares the final files used in Power BI. Most of the work here is not statistical modeling; it is making sure the dashboard has clean, stable tables with the columns Power BI expects.

The early-warning examples use synthetic demo flags so the dashboard can show how risk monitoring would look.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# keeping imports together so the notebook is easier to rerun
# 1) Import libraries

import pandas as pd
from pathlib import Path

In [ ]:
# 2) Set file paths

# Change only BASE_DIR if the project folder is in a different place
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Practicum/Demo File")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
DASHBOARD_DIR = BASE_DIR / "data" / "dashboard_ready"
POWERBI_DIR = BASE_DIR / "data" / "powerbi_replacement"

DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

print("Processed folder:", PROCESSED_DIR)
print("Dashboard-ready folder:", DASHBOARD_DIR)
print("Power BI replacement folder:", POWERBI_DIR)


In [ ]:
# first I load the files and then check the shape/columns before trusting anything
# 3) Helper function to load files safely

def load_csv_if_exists(file_name):
    """
    Small helper so I can quickly see if any expected output file is missing.
    """
    file_path = PROCESSED_DIR / file_name

    if file_path.exists():
        print("Loaded:", file_name)
        return pd.read_csv(file_path)
    else:
        print("Missing:", file_name)
        return pd.DataFrame()

In [ ]:
# duplicates can quietly inflate store totals, so I remove them at the store-date/month level
# 4) Load cleaned datasets and analysis outputs

clean_daily = load_csv_if_exists("clean_daily_master.csv")
clean_monthly = load_csv_if_exists("clean_monthly_kpi.csv")
kpi_dictionary = load_csv_if_exists("kpi_dictionary.csv")

kpi_framework = load_csv_if_exists("kpi_framework.csv")
store_performance = load_csv_if_exists("store_performance_summary.csv")
monthly_analysis = load_csv_if_exists("monthly_analysis_dataset.csv")
kpi_relationship_summary = load_csv_if_exists("kpi_relationship_summary.csv")

store_classification = load_csv_if_exists("store_classification.csv")
opportunity_diagnostic = load_csv_if_exists("upselling_opportunity_diagnostic.csv")

store_type_segment = load_csv_if_exists("segment_store_type_summary.csv")
zone_segment = load_csv_if_exists("segment_market_zone_summary.csv")
size_segment = load_csv_if_exists("segment_store_size_summary.csv")

promotion_month_summary = load_csv_if_exists("promotion_month_summary.csv")
inventory_month_summary = load_csv_if_exists("inventory_month_summary.csv")

monthly_forecast_results = load_csv_if_exists("monthly_forecasting_model_results.csv")
monthly_forecast_predictions = load_csv_if_exists("monthly_forecast_test_predictions.csv")

daily_forecast_results = load_csv_if_exists("daily_forecasting_model_results.csv")
daily_forecast_predictions = load_csv_if_exists("daily_forecast_test_predictions.csv")
daily_30_day_forecast = load_csv_if_exists("daily_30_day_future_forecast.csv")

rf_feature_importance = load_csv_if_exists("rf_feature_importance.csv")
xgb_feature_importance = load_csv_if_exists("xgb_feature_importance.csv")

# Small cleanup for dashboard display.
# Power BI visuals are easier to maintain when store/location fields are consistent.
if not opportunity_diagnostic.empty:
    opportunity_diagnostic = opportunity_diagnostic.copy()
    if "Store_Name" in opportunity_diagnostic.columns and "Store_Loc" not in opportunity_diagnostic.columns:
        opportunity_diagnostic = opportunity_diagnostic.rename(columns={"Store_Name": "Store_Loc"})

    # Make sure Performance_Category exists because some Power BI queries/visuals expect it.
    # If it is missing, bring it from store_classification using Store_ID.
    if "Performance_Category" not in opportunity_diagnostic.columns and not store_classification.empty:
        category_lookup = store_classification[["Store_ID", "Performance_Category"]].drop_duplicates()
        opportunity_diagnostic = opportunity_diagnostic.merge(category_lookup, on="Store_ID", how="left")

    # Shorter text for the dashboard table. The original full text was too long for the visual.
    def shorten_strength(text):
        text = str(text)
        parts = []
        if "accessory profit" in text.lower():
            parts.append("High profit/activation")
        if "revenue per device" in text.lower() or "revenue per box" in text.lower():
            parts.append("High acc. revenue/box")
        if "productivity" in text.lower() or "ppd" in text.lower():
            parts.append("High PPD/hour")
        if len(parts) == 0:
            return "No major strength"
        return "; ".join(parts)

    def shorten_opportunity(text):
        text = str(text)
        parts = []
        if "profit per activation" in text.lower():
            parts.append("Improve profit/activation")
        if "revenue per device" in text.lower() or "revenue per box" in text.lower():
            parts.append("Improve acc. revenue/box")
        if "productivity" in text.lower() or "ppd" in text.lower():
            parts.append("Improve PPD/hour")
        if "maintain" in text.lower() and len(parts) == 0:
            return "Maintain and monitor"
        if len(parts) == 0:
            return "Monitor"
        return "; ".join(parts)

    if "Store_Strength" in opportunity_diagnostic.columns:
        opportunity_diagnostic["Store_Strength"] = opportunity_diagnostic["Store_Strength"].apply(shorten_strength)
    if "Store_Opportunity" in opportunity_diagnostic.columns:
        opportunity_diagnostic["Store_Opportunity"] = opportunity_diagnostic["Store_Opportunity"].apply(shorten_opportunity)

    # Keep important columns in a clean order.
    opp_cols = [
        "Store_ID", "Store_Loc", "Market_Zone", "Store_Type", "Store_Size",
        "Performance_Category", "Store_Strength", "Store_Opportunity"
    ]
    opportunity_diagnostic = opportunity_diagnostic[[c for c in opp_cols if c in opportunity_diagnostic.columns]]

    display(opportunity_diagnostic.head())


In [ ]:
# dates were messy in the raw demo files, so I avoid assuming one perfect date format
# 5) Convert date columns

# Power BI usually reads dates fine, but I still clean them here so the export is consistent.

date_tables = [
    (clean_daily, "Date"),
    (clean_monthly, "Month"),
    (monthly_analysis, "Month"),
    (monthly_forecast_predictions, "ds"),
    (daily_forecast_predictions, "ds"),
    (daily_30_day_forecast, "ds")
]

for table, date_col in date_tables:
    if not table.empty and date_col in table.columns:
        table[date_col] = pd.to_datetime(table[date_col])

In [ ]:
# 6) Create executive monthly trend table

# This table supports the dashboard overview page.
# It summarizes market-level performance by month.

clean_daily["Date"] = pd.to_datetime(clean_daily["Date"])
clean_daily["Month"] = clean_daily["Date"].dt.to_period("M").dt.to_timestamp()

executive_monthly_trend = (
    clean_daily
    .groupby("Month", as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        New_Activation=("New_Activation", "sum"),
        Upgrade_SOR=("Upgrade_SOR", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum"),
        Promotion_Days=("Promotion_Flag", "sum"),
        Inventory_Issue_Days=("Inventory_Issue_Flag", "sum"),
        Local_Event_Days=("Local_Event_Flag", "sum")
    )
)

executive_monthly_trend["Activation_MoM_Change"] = executive_monthly_trend["Total_Activation"].diff()
executive_monthly_trend["Activation_MoM_Change_Pct"] = executive_monthly_trend["Total_Activation"].pct_change() * 100

display(executive_monthly_trend.head())

In [ ]:
# 7) Create executive KPI card table

# This makes it easier to create KPI cards in Power BI.

executive_kpi_cards = pd.DataFrame({
    "KPI_Name": [
        "Total Activation",
        "New Activation",
        "Account Gross",
        "Accessory Profit",
        "Total PPD",
        "Average PPD per Hour",
        "Accessory Profit per Activation"
    ],
    "KPI_Value": [
        clean_daily["Total_Activation"].sum(),
        clean_daily["New_Activation"].sum(),
        clean_daily["Account_Gross"].sum(),
        clean_daily["Accessory_Profit"].sum(),
        store_performance["PPD"].sum() if "PPD" in store_performance.columns else None,
        store_performance["PPD_per_Hour"].mean() if "PPD_per_Hour" in store_performance.columns else None,
        clean_daily["Accessory_Profit"].sum() / clean_daily["Total_Activation"].sum()
    ],
    "KPI_Type": [
        "Activity",
        "Activity",
        "Financial",
        "Financial",
        "Activity",
        "Efficiency",
        "Efficiency"
    ]
})

display(executive_kpi_cards)

In [ ]:
# 8) Create store rank table

# This table supports top/bottom store views.

store_rank_table = store_performance.copy()

rank_cols = [
    "Total_Activation",
    "Account_Gross",
    "Accessory_Profit",
    "PPD",
    "PPD_per_Hour",
    "Accessory_Profit_per_Activation",
    "Accessory_Revenue_per_Box"
]

for col in rank_cols:
    if col in store_rank_table.columns:
        store_rank_table[col + "_Rank"] = store_rank_table[col].rank(ascending=False, method="min")

display(store_rank_table.head())

In [ ]:
# 9) Create early warning table

# This creates a simple risk table using recent monthly performance.
# It is useful for the dashboard forecasting / monitoring page.

monthly_store = (
    clean_daily
    .groupby(["Month", "Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size"], as_index=False)
    .agg(Total_Activation=("Total_Activation", "sum"))
)

warning_rows = []

for store_id in monthly_store["Store_ID"].unique():
    temp = monthly_store[monthly_store["Store_ID"] == store_id].sort_values("Month").copy()

    store_name = temp["Store_Name"].iloc[0]
    market_zone = temp["Market_Zone"].iloc[0]
    store_type = temp["Store_Type"].iloc[0]
    store_size = temp["Store_Size"].iloc[0]

    latest_month = temp["Month"].max()
    latest_activation = temp.loc[temp["Month"] == latest_month, "Total_Activation"].iloc[0]

    historical_avg = temp["Total_Activation"].mean()
    recent_3_avg = temp["Total_Activation"].tail(3).mean()
    recent_6_avg = temp["Total_Activation"].tail(6).mean()

    recent_3_diff_pct = ((recent_3_avg - historical_avg) / historical_avg) * 100
    recent_6_diff_pct = ((recent_6_avg - historical_avg) / historical_avg) * 100

    if recent_3_diff_pct <= -10 and recent_6_diff_pct <= -10:
        risk_level = "Strong Decline Risk"
    elif recent_3_diff_pct <= -10:
        risk_level = "Warning"
    elif recent_3_diff_pct <= -5:
        risk_level = "Watch"
    else:
        risk_level = "No Major Concern"

    warning_rows.append({
        "Store_ID": store_id,
        "Store_Loc": store_name,
        "Market_Zone": market_zone,
        "Store_Type": store_type,
        "Store_Size": store_size,
        "Latest_Month": latest_month,
        "Latest_Activation": latest_activation,
        "Historical_Monthly_Avg": round(historical_avg, 2),
        "Recent_3_Month_Avg": round(recent_3_avg, 2),
        "Recent_6_Month_Avg": round(recent_6_avg, 2),
        "Recent_3_Diff_Pct": round(recent_3_diff_pct, 2),
        "Recent_6_Diff_Pct": round(recent_6_diff_pct, 2),
        "Risk_Level": risk_level
    })

early_warning_table = pd.DataFrame(warning_rows)

# 9b. Add demo risk cases for dashboard storytelling

# Note:
# The public dataset is synthetic, and the original early-warning output
# did not produce decline-risk stores. I am only adjusting the derived
# early-warning output table so the dashboard can demonstrate how Watch,
# Warning, and Strong Decline Risk flags would appear.
#
# I am NOT changing the raw daily data or monthly KPI data.

risk_updates = {
    "STORE010": {
        "Recent_3_Diff_Pct": -6.8,
        "Recent_6_Diff_Pct": -3.2,
        "Risk_Level": "Watch"
    },
    "STORE017": {
        "Recent_3_Diff_Pct": -12.4,
        "Recent_6_Diff_Pct": -6.5,
        "Risk_Level": "Warning"
    },
    "STORE020": {
        "Recent_3_Diff_Pct": -18.7,
        "Recent_6_Diff_Pct": -13.1,
        "Risk_Level": "Strong Decline Risk"
    }
}

for store_id, updates in risk_updates.items():
    for col, value in updates.items():
        early_warning_table.loc[early_warning_table["Store_ID"] == store_id, col] = value

# Sort so the most concerning stores appear first in the saved output.
early_warning_table = early_warning_table.sort_values("Recent_3_Diff_Pct").reset_index(drop=True)

display(early_warning_table.head(10))


In [ ]:
# 10) Add risk-based follow-up guidance for dashboard

# This is rule-based follow-up guidance, not a causal recommendation.
# It helps dashboard users know which stores may need closer review.

def add_follow_up_guidance(row):
    if row["Risk_Level"] == "Strong Decline Risk":
        return "High priority review: compare recent activity, inventory, and store operations."
    elif row["Risk_Level"] == "Warning":
        return "Monitor closely and review recent activation drivers."
    elif row["Risk_Level"] == "Watch":
        return "Track next month and compare with historical trend."
    else:
        return "Continue regular monitoring."

early_warning_table["Suggested_Follow_Up"] = early_warning_table.apply(add_follow_up_guidance, axis=1)

# Keeping this old column name too because the current Power BI visual already expects it.
# Later, we can rename the display label inside Power BI.
early_warning_table["Recommended_Action"] = early_warning_table["Suggested_Follow_Up"]

# Reorder columns so Power BI sees a stable table structure.
early_warning_cols = [
    "Store_ID", "Store_Loc", "Market_Zone", "Store_Type", "Store_Size",
    "Latest_Month", "Latest_Activation",
    "Historical_Monthly_Avg", "Recent_3_Month_Avg", "Recent_6_Month_Avg",
    "Recent_3_Diff_Pct", "Recent_6_Diff_Pct",
    "Risk_Level", "Recommended_Action", "Suggested_Follow_Up"
]

early_warning_table = early_warning_table[[c for c in early_warning_cols if c in early_warning_table.columns]]

display(early_warning_table.head(10))
print(early_warning_table["Risk_Level"].value_counts())


In [ ]:
# 11) Create forecasting summary table

best_monthly = monthly_forecast_results.sort_values("MAE").head(1).copy()
best_daily = daily_forecast_results.sort_values("MAE").head(1).copy()

forecast_summary = pd.concat([best_monthly, best_daily], ignore_index=True)

display(forecast_summary)

In [ ]:
# saving this output so the next notebook / Power BI can reuse the same table
# 12) Save dashboard-ready CSV files

dashboard_tables = {
    "dashboard_clean_daily_master.csv": clean_daily,
    "dashboard_clean_monthly_kpi.csv": clean_monthly,
    "dashboard_executive_monthly_trend.csv": executive_monthly_trend,
    "dashboard_executive_kpi_cards.csv": executive_kpi_cards,
    "dashboard_store_performance.csv": store_performance,
    "dashboard_store_rank_table.csv": store_rank_table,
    "dashboard_store_classification.csv": store_classification,
    "dashboard_opportunity_diagnostic.csv": opportunity_diagnostic,
    "dashboard_kpi_framework.csv": kpi_framework,
    "dashboard_kpi_dictionary.csv": kpi_dictionary,
    "dashboard_kpi_relationship_summary.csv": kpi_relationship_summary,
    "dashboard_store_type_segment.csv": store_type_segment,
    "dashboard_market_zone_segment.csv": zone_segment,
    "dashboard_store_size_segment.csv": size_segment,
    "dashboard_promotion_month_summary.csv": promotion_month_summary,
    "dashboard_inventory_month_summary.csv": inventory_month_summary,
    "dashboard_monthly_forecast_results.csv": monthly_forecast_results,
    "dashboard_monthly_forecast_predictions.csv": monthly_forecast_predictions,
    "dashboard_daily_forecast_results.csv": daily_forecast_results,
    "dashboard_daily_forecast_predictions.csv": daily_forecast_predictions,
    "dashboard_daily_30_day_forecast.csv": daily_30_day_forecast,
    "dashboard_rf_feature_importance.csv": rf_feature_importance,
    "dashboard_xgb_feature_importance.csv": xgb_feature_importance,
    "dashboard_early_warning_table.csv": early_warning_table,
    "dashboard_forecast_summary.csv": forecast_summary
}

for file_name, table in dashboard_tables.items():
    if not table.empty:
        table.to_csv(DASHBOARD_DIR / file_name, index=False)

print("Dashboard CSV files saved in:", DASHBOARD_DIR)
print("Early warning risk levels saved:")
print(early_warning_table["Risk_Level"].value_counts())


In [ ]:
# 13) Save one Excel workbook for Power BI

# I also save everything into one workbook because it is easier to connect
# Power BI to one file instead of many separate CSVs.

dashboard_workbook_path = DASHBOARD_DIR / "Retail_Store_Dashboard_Ready_Data.xlsx"


with pd.ExcelWriter(dashboard_workbook_path, engine="openpyxl") as writer:
    for sheet_name, table in {
        "Clean_Daily_Master": clean_daily,
        "Clean_Monthly_KPI": clean_monthly,
        "Executive_Monthly_Trend": executive_monthly_trend,
        "Executive_KPI_Cards": executive_kpi_cards,
        "Store_Performance": store_performance,
        "Store_Rank_Table": store_rank_table,
        "Store_Classification": store_classification,
        "Opportunity_Diagnostic": opportunity_diagnostic,
        "KPI_Framework": kpi_framework,
        "KPI_Dictionary": kpi_dictionary,
        "KPI_Relationships": kpi_relationship_summary,
        "Store_Type_Segment": store_type_segment,
        "Market_Zone_Segment": zone_segment,
        "Store_Size_Segment": size_segment,
        "Promotion_Summary": promotion_month_summary,
        "Inventory_Summary": inventory_month_summary,
        "Monthly_Forecast_Results": monthly_forecast_results,
        "Monthly_Forecast_Test": monthly_forecast_predictions,
        "Daily_Forecast_Results": daily_forecast_results,
        "Daily_Forecast_Test": daily_forecast_predictions,
        "Daily_30_Day_Forecast": daily_30_day_forecast,
        "RF_Feature_Importance": rf_feature_importance,
        "XGB_Feature_Importance": xgb_feature_importance,
        "Early_Warning": early_warning_table,
        "Forecast_Summary": forecast_summary
    }.items():
        if not table.empty:
            table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Saved dashboard workbook:")
print(dashboard_workbook_path)

In [ ]:
# 14) Final validation

print("Dashboard Data Preparation Summary")
print("----------------------------------")

print("Dashboard folder:", DASHBOARD_DIR)
print("Workbook created:", dashboard_workbook_path.exists())

print("\nMain tables:")
print("Clean daily:", clean_daily.shape)
print("Clean monthly:", clean_monthly.shape)
print("Store performance:", store_performance.shape)
print("Store classification:", store_classification.shape)
print("Early warning:", early_warning_table.shape)
print("Daily forecast:", daily_30_day_forecast.shape)

print("\nBest forecast models:")
display(forecast_summary)

print("\nEarly warning preview:")
display(early_warning_table.sort_values("Recent_3_Diff_Pct").head())

In [ ]:
# 15) Save Power BI replacement workbook

# This workbook uses the same sheet names that the existing Power BI dashboard expects.
# Use this file when refreshing the public/demo dashboard.

powerbi_workbook_path = POWERBI_DIR / "Metro_TMobile_Dashboard_Demo_Replacement.xlsx"

# Make day-wise fact tables for the existing dashboard model.
total_activation_daywise = clean_daily[["Date", "Store_ID", "Store_Name", "Total_Activation"]].rename(columns={"Store_Name": "Store_Loc"})
new_activation_daywise = clean_daily[["Date", "Store_ID", "Store_Name", "New_Activation"]].rename(columns={"Store_Name": "Store_Loc"})
upgrade_sor_daywise = clean_daily[["Date", "Store_ID", "Store_Name", "Upgrade_SOR"]].rename(columns={"Store_Name": "Store_Loc"})
accessory_profit_daywise = clean_daily[["Date", "Store_ID", "Store_Name", "Accessory_Profit"]].rename(columns={"Store_Name": "Store_Loc"})
account_gross_daywise = clean_daily[["Date", "Store_ID", "Store_Name", "Account_Gross"]].rename(columns={"Store_Name": "Store_Loc"})

# Dimension tables.
dim_store = store_classification[[
    "Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size",
    "Volume_Score", "Efficiency_Score", "Performance_Category"
]].rename(columns={"Store_Name": "Store_Loc"})

dim_date = pd.DataFrame({
    "Date": pd.date_range(clean_daily["Date"].min(), clean_daily["Date"].max(), freq="D")
})
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month Number"] = dim_date["Date"].dt.month
dim_date["Month Name"] = dim_date["Date"].dt.strftime("%b")
dim_date["Month-Year"] = dim_date["Date"].dt.strftime("%b %Y")
dim_date["Month Start"] = dim_date["Date"].dt.to_period("M").dt.to_timestamp()
dim_date["Month Sort"] = dim_date["Date"].dt.year * 100 + dim_date["Date"].dt.month
dim_date["Quarter"] = "Q" + dim_date["Date"].dt.quarter.astype(str)

dim_month = pd.DataFrame({
    "Month": pd.date_range(
        clean_daily["Date"].min().to_period("M").to_timestamp(),
        clean_daily["Date"].max().to_period("M").to_timestamp(),
        freq="MS"
    )
})
dim_month["Month Number"] = dim_month["Month"].dt.month
dim_month["Month Name"] = dim_month["Month"].dt.strftime("%b")
dim_month["Month-Year"] = dim_month["Month"].dt.strftime("%b %Y")
dim_month["Month Sort"] = dim_month["Month"].dt.year * 100 + dim_month["Month"].dt.month

# KPI master uses Store_Loc for consistency with the existing dashboard.
kpi_master = clean_monthly.rename(columns={"Store_Name": "Store_Loc"}).copy()

with pd.ExcelWriter(powerbi_workbook_path, engine="openpyxl") as writer:
    total_activation_daywise.to_excel(writer, sheet_name="Total Activation Daywise", index=False)
    new_activation_daywise.to_excel(writer, sheet_name="New Activation Daywise", index=False)
    upgrade_sor_daywise.to_excel(writer, sheet_name="Upgrade SOR Daywise", index=False)
    accessory_profit_daywise.to_excel(writer, sheet_name="Accessory Profit Daywise", index=False)
    account_gross_daywise.to_excel(writer, sheet_name="Account Gross Daywise", index=False)

    dim_store.to_excel(writer, sheet_name="Dim_Store", index=False)
    dim_date.to_excel(writer, sheet_name="Dim_Date", index=False)
    dim_month.to_excel(writer, sheet_name="Dim_Month", index=False)

    kpi_master.to_excel(writer, sheet_name="KPI Master", index=False)
    executive_monthly_trend.to_excel(writer, sheet_name="Executive_Monthly_Trend", index=False)

    # Existing Power BI forecast page expects these exact sheet names.
    monthly_forecast_predictions.to_excel(writer, sheet_name="Market_Forecast_Trend", index=False)
    monthly_forecast_predictions.to_excel(writer, sheet_name="Store_Monthly_Forecast", index=False)
    early_warning_table.to_excel(writer, sheet_name="Store_Early_Warning", index=False)

    # Page 3 diagnostic table.
    opportunity_diagnostic.to_excel(writer, sheet_name="Opportunity_Diagnostic", index=False)

    daily_30_day_forecast.to_excel(writer, sheet_name="Daily_30_Day_Forecast", index=False)
    daily_forecast_results.to_excel(writer, sheet_name="Daily_Forecast_Results", index=False)

    last_refresh = pd.DataFrame({"LastRefresh": [pd.Timestamp.now().strftime("%b %d, %Y %I:%M %p")]})
    last_refresh.to_excel(writer, sheet_name="Last_Refresh", index=False)

print("Saved Power BI replacement workbook:")
print(powerbi_workbook_path)
print("\nStore_Early_Warning columns:", early_warning_table.columns.tolist())
print("Opportunity_Diagnostic columns:", opportunity_diagnostic.columns.tolist())
